In [2]:
import requests
from bs4 import BeautifulSoup
import html2text
import argparse
import re
import sys
from urllib.parse import urlparse


def is_content_element(element):
    if element.name is None:
        return False

    skip_patterns = [
        'header', 'footer', 'nav', 'menu', 'sidebar', 'banner',
        'advertisement', 'cookie', 'popup', 'modal', 'social',
        'comment', 'widget', 'toolbar', 'masthead'
    ]

    attrs = []
    if element.has_attr('class'):
        attrs.extend(element['class'])
    if element.has_attr('id'):
        attrs.append(element['id'])

    attrs = [attr.lower() for attr in attrs]

    for attr in attrs:
        for pattern in skip_patterns:
            if pattern in attr:
                return False

    if element.has_attr('role'):
        role = element['role'].lower()
        skip_roles = ['navigation', 'banner', 'complementary', 'contentinfo']
        if role in skip_roles:
            return False

    return True


def extract_content(soup):
    main_elements = soup.find_all(['main', 'article'])
    if main_elements:
        return main_elements

    content_divs = []
    for div in soup.find_all('div'):
        attrs = []
        if div.has_attr('class'):
            attrs.extend(div['class'])
        if div.has_attr('id'):
            attrs.append(div['id'])

        attrs = [attr.lower() for attr in attrs]
        if any('content' in attr for attr in attrs):
            content_divs.append(div)

    if content_divs:
        return content_divs

    body = soup.find('body')
    if body:
        content_elements = []
        for elem in body.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'ul', 'ol', 'blockquote']):
            if is_content_element(elem) and len(elem.get_text(strip=True)) > 20:  # Require some text
                content_elements.append(elem)
        return content_elements

    paragraphs = soup.find_all('p')
    if paragraphs:
        return [p for p in paragraphs if len(p.get_text(strip=True)) > 50]

    return [body] if body else []


def scrape_to_markdown(url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Referer': 'https://www.google.com/',
            'DNT': '1',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
        }

        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        for element in soup(['script', 'style', 'noscript', 'svg', 'iframe']):
            element.decompose()

        content_elements = extract_content(soup)

        if not content_elements:
            return "No content found on the page."

        title = ""
        title_elem = soup.find('title')
        if title_elem:
            title = f"# {title_elem.get_text().strip()}\n\n"

        converter = html2text.HTML2Text()
        converter.ignore_links = False
        converter.ignore_images = False
        converter.ignore_tables = False
        converter.body_width = 0

        markdown_content = title

        for element in content_elements:
            element_html = str(element)
            md_part = converter.handle(element_html)

            md_part = re.sub(r'\n{3,}', '\n\n', md_part)
            markdown_content += md_part + "\n\n"

        markdown_content = re.sub(r'\n{3,}', '\n\n', markdown_content)

        return markdown_content.strip()

    except requests.exceptions.RequestException as e:
        return f"Error: Failed to retrieve the webpage: {str(e)}"
    except Exception as e:
        return f"Error: An unexpected error occurred: {str(e)}"


def get_output_filename(url):
    parsed = urlparse(url)
    domain = parsed.netloc.replace("www.", "")
    path = parsed.path.strip("/").replace("/", "_")
    if not path:
        path = "index"
    return f"{domain}_{path}.md"


In [15]:
url = "https://sylabusy.agh.edu.pl/pl/1/2/21/0/0/16"
output_file = f"{url[8:].replace('.', '-').replace('/', '_')}.md"

markdown_content = scrape_to_markdown(url)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"Content saved to {output_file}")

Content saved to sylabusy-agh-edu-pl_pl_1_2_21_0_0_16.md
